In [0]:
import requests
import pandas as pd

session = requests.Session()

p_url = "https://pokeapi.co/api/v2/pokemon/"

all_p_data = []
all_s_data = []
all_l_data = []

for p in range(1, 20000):

    p_response = session.get(f"{p_url}{p}")

    if p_response.status_code == 200:
        p_data = p_response.json()
        # getting url
        species_url = p_data["species"]["url"]
        location_url = p_data["location_area_encounters"]


        # json alle data
        
        s_data = session.get(species_url).json()
        l_data = session.get(location_url).json()
        # Lege directory maken 
        stats = {}
        for s in p_data["stats"]: # Runs through all data in p_data till s doesnt find anything
            s_key = s["stat"]["name"] 
            s_value = s["base_stat"] 
            stats[s_key] = s_value  # Value saved in stats[key]

        # types uitpkakken zie hierboven
        types = {}
        for t in p_data["types"]:
            s_key = t["slot"] 
            t_value = t["type"]["name"] 
            types[s_key] = t_value

        egg_list = []
        for e in s_data["egg_groups"]:
            egg_list.append(e["name"])    
        eggs = ", ".join(egg_list)
        


        # Append pokemon stats
        all_p_data.append({
            "id": p_data.get("id", {}),
            "pokemon": p_data.get("name", {}).capitalize(),
            "height": p_data.get("height", {}),
            "order": p_data.get("order", {}),
            "weight": p_data.get("weight", {}),
            "base-experience": p_data.get("base_experience", {}),
            "hp": stats.get("hp", {}),
            "attack": stats.get("attack", {}),
            "defense": stats.get("defense", {}),
            "speed": stats.get("speed", {}),
            "special-attack": stats.get("special-attack", {}),
            "special-defense": stats.get("special-defense", {}),
        })
        
        # Append species data
        all_s_data.append({
            "id": (s_data.get("id") or {}),
            "hapiness": s_data.get("base_happiness"),
            "capture-rate": s_data.get("capture_rate"),
            "gender-rate": s_data.get("gender_rate"),
            "gender-difference": s_data.get("has_gender_differences"),
            "baby": s_data.get("is_baby"),
            "legendary": s_data.get("is_legendary"),
            "mythical": s_data.get("is_mythical"),
            "color": (s_data.get("color") or {}).get("name"),
            "generation": (s_data.get("generation") or {}).get("name"),
            "growth-rate": (s_data.get("growth_rate") or {}).get("name"),
            "shape": (s_data.get("shape") or {}).get("name"),
            "habitat": (s_data.get("habitat") or {}).get("name"),
        })

        # append random data
        all_l_data.append({
            "id": (s_data.get("id") or {}),
            "egg-group": eggs,
            "types": ", ".join(types.values()),
            "is-default": p_data.get("is_default", {}),

        })




    else:
        break

#Filter override
#df = df.sort_values(by="weight", ascending=False) sorteren op hoogte
# print(df["height"].describe())

sp_df = spark.createDataFrame(all_p_data)
sp_dp = spark.createDataFrame(all_s_data)
sp_da = spark.createDataFrame(all_l_data)

sp_df.write.format("delta").mode("overwrite").saveAsTable("pokemon_stats")
sp_dp.write.format("delta").mode("overwrite").saveAsTable("pokemon_species")
sp_da.write.format("delta").mode("overwrite").saveAsTable("pokemon_eggs")

# display(sp_df)
# display(sp_dp)
# display(sp_da)


In [0]:

result = spark.sql("SELECT * FROM pokemon_stats")
display(result)